# 06 - Deployment
**CRISP-DM phase:** Deployment

The selected models are delivered as **Vehicle Safety Intelligence**, a working web application with three human-review workflows:

| Workflow | Models used | Answers |
|---|---|---|
| Complaint triage | U1 + U2 | Which component group(s)? Does it describe a crash, fire, injury or death? |
| Crash severity | compact S1 + S2 | How likely is injury, and serious or fatal injury, for this crash? Which inputs drive the estimate? |
| Vehicle lookup | linkage tables | How does this make/model/year compare once complaints are adjusted for on-road exposure? |

| Component | Location | Hosting |
|---|---|---|
| Model API (FastAPI) | `app/api/main.py` | Render: https://vehicle-safety-api.onrender.com |
| Web interface (HTML/JS) | `app/web/` | Vercel: https://vehicle-safety-intelligence-web.vercel.app |
| Automated tests | `tests/test_api.py` | run locally before each release |

> **Initial plan:** a Streamlit dashboard calling the models directly (proposal and the first version of this notebook).
> **What we found:** a dashboard that holds the models itself can only be used through that dashboard, and free Streamlit hosting suits demonstrations rather than a service other tools could call.
> **Decision:** separate the **model API** (FastAPI, on Render) from the **interface** (static web page, on Vercel). Any system, such as a complaint-intake form or a case-management tool, can call the same API, and the interface stays lightweight.

## 6.0 Setup
The API is loaded in-process with FastAPI's test client, so every request below runs exactly the code that is deployed.

In [1]:
from pathlib import Path
import json, subprocess, sys

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
API_DIR = PROJECT_ROOT / "app" / "api"
sys.path.insert(0, str(API_DIR))
from fastapi.testclient import TestClient
import main as api

client = TestClient(api.app)
ART = API_DIR / "artifacts"

## 6.1 Deployment artifacts
Only what the application needs is deployed: the two text models (each with its TF-IDF vocabulary), the two compact crash models, the crash-form definition, the vehicle profiles and a model card of test results. Raw data (6 GB) stays out of the deployment.

In [2]:
what = {"text_U1.joblib": "U1 routing: TF-IDF vocabulary + 19 one-vs-rest Linear SVMs + thresholds",
        "text_U2.joblib": "U2 serious-incident flag: TF-IDF vocabulary + Linear SVM + threshold",
        "crash_y_injury.joblib": "compact S1 pipeline (30 inputs) + threshold",
        "crash_y_serious.joblib": "compact S2 pipeline (30 inputs) + threshold",
        "crash_form.json": "the 30 crash inputs, their official code labels and valid ranges",
        "vehicle_profiles.parquet": "make/model/year complaint and CRSS exposure profiles",
        "model_card.json": "headline test metrics shown on the Overview page"}
pd.DataFrame([{"file": f.name, "MB": round(f.stat().st_size / 1e6, 2), "contents": what.get(f.name, "")}
              for f in sorted(ART.iterdir())])

,file,MB,contents
0,crash_form.json,0.01,"the 30 crash inputs, their official code label..."
1,crash_y_injury.joblib,0.48,compact S1 pipeline (30 inputs) + threshold
2,crash_y_serious.joblib,0.38,compact S2 pipeline (30 inputs) + threshold
3,model_card.json,0.00,headline test metrics shown on the Overview page
4,text_U1.joblib,47.78,U1 routing: TF-IDF vocabulary + 19 one-vs-rest...
5,text_U2.joblib,6.17,U2 serious-incident flag: TF-IDF vocabulary + ...
6,vehicle_profiles.parquet,0.44,make/model/year complaint and CRSS exposure pr...


## 6.2 Complaint triage through the API

In [3]:
narrative = ("I was rear-ended at a stop light and the air bags did not deploy. My head hit the steering wheel "
             "and I was taken to the hospital by ambulance. A police report was filed.")
r = client.post("/api/complaint/analyze", json={"text": narrative}).json()
print("Serious-incident flag:", r["serious_incident"]["flagged"], "| score", r["serious_incident"]["score"],
      "| threshold", r["serious_incident"]["threshold"])
print("Evidence:", [e["term"] for e in r["serious_incident"]["evidence"]])
pd.DataFrame([{"component": c["component"], "score": c["score"], "threshold": c["threshold"],
               "recommended": c["recommended"], "evidence": ", ".join(e["term"] for e in c["evidence"][:5])}
              for c in r["components"]])

Serious-incident flag: True | score 3.048 | threshold -0.1775
Evidence: ['hit', 'was rear', 'hospital', 'bags did', 'rear ended', 'ended', 'police', 'police report']


,component,score,threshold,recommended,evidence
0,AIR BAGS,1.7888,0.0105,True,"bags, air bags, air, deploy, not deploy"
1,STEERING,-0.4906,-0.1952,False,"steering, the steering, wheel, police, hospital"
2,DRIVER ASSISTANCE,-1.0648,-0.6512,False,"rear ended, wheel and, light, filed, was rear"
3,EQUIPMENT,-1.0672,-0.5956,False,"the air, my head, light and, wheel and, air"
4,SEAT BELTS,-0.7771,-0.2548,False,"head hit, was rear, light and, my head, hit the"


**Interpretation.** The narrative is routed to **AIR BAGS** and flagged for priority review. Each decision comes with the terms that pushed it (term weight multiplied by its TF-IDF value in this narrative), so a reviewer can see why.

### Round-trip check
The score returned by the API must equal the score from loading the saved model directly; this confirms the deployed service uses the validated model unchanged.

In [4]:
u2 = joblib.load(ART / "text_U2.joblib")
direct = float(u2["model"].decision_function(u2["vectorizer"].transform([narrative]))[0])
print(f"direct {direct:.6f} | via API {r['serious_incident']['score']:.6f} | equal: {abs(direct - r['serious_incident']['score']) < 1e-4}")

direct 3.047994 | via API 3.048000 | equal: True


## 6.3 Crash severity through the API
The form exposes 30 police-report inputs; any may be left blank. Two contrasting crashes are assessed.

In [5]:
form = {f["name"]: f for f in client.get("/api/crash/form").json()["fields"]}
code_of = lambda field, label: next(o["value"] for o in form[field]["options"] if o["label"] == label)
severe = {"first_harmful_event": code_of("first_harmful_event", "Rollover/Overturn"), "vehicles": 1, "occupants": 2,
          "max_travel_speed": 75, "speed_limit": 55, "rollovers": 1, "unrestrained_occupants": 1, "youngest_person_age": 19}
minor = {"first_harmful_event": code_of("first_harmful_event", "Motor Vehicle In-Transport"),
         "manner_of_collision": code_of("manner_of_collision", "Front-to-Rear"), "vehicles": 2, "occupants": 2,
         "max_travel_speed": 5, "speed_limit": 25, "rollovers": 0, "unrestrained_occupants": 0}
for name, inputs in [("severe rollover", severe), ("low-speed rear-end", minor)]:
    out = client.post("/api/crash/assess", json=inputs).json()
    print(f"\n{name}: injury {out['injury_crash']['probability']:.1%} | serious/fatal {out['serious_or_fatal_crash']['probability']:.1%}")
    print("  warning:", out["warning"])
    print("  main factors (serious):", [(f["label"], f["effect"]) for f in out["serious_or_fatal_crash"]["factors"][:4]])


severe rollover: injury 97.8% | serious/fatal 50.1%
  main factors (serious): [('Occupants not using a restraint', 0.1706), ('Vehicles that rolled over', 0.1624), ('Speed limit (mph)', 0.1372), ('Highest travel speed (mph)', 0.1114)]

low-speed rear-end: injury 36.3% | serious/fatal 0.6%
  main factors (serious): [('Highest travel speed (mph)', -0.0111), ('First harmful event', -0.003), ('Speed limit (mph)', -0.0023), ('Manner of collision', -0.0021)]


**Interpretation.** The severe rollover scores far higher than the low-speed rear-end collision on both targets. The **factors** are a local explanation: for each input provided, the change in probability when that input is set back to unknown. Every response states how many inputs were missing, and that probabilities are relative to the CRSS sample, which over-represents injury crashes.

## 6.4 Vehicle lookup through the API

In [6]:
p = client.get("/api/vehicles/profile", params={"make": "FORD", "model": "ESCAPE", "year": 2017}).json()
print(json.dumps({k: p[k] for k in ("vehicle", "integrated")}, indent=1))
print("Top components:", dict(list(p["complaints"]["by_component"].items())[:4]))
pd.DataFrame(client.get("/api/vehicles/top", params={"limit": 10}).json())

{
 "vehicle": {
  "make": "FORD",
  "model": "ESCAPE",
  "model_year": 2017
 },
 "integrated": {
  "complaints_per_10k_crash_involved": 287.1357,
  "percentile_among_vehicles": 0.9671,
  "reliable": true
 }
}
Top components: {'ENGINE': 1125, 'POWER TRAIN': 262, 'ELECTRICAL': 68, 'WHEELS': 61}


,make,model,model_year,complaints,complaints_per_10k_crash_involved,serious_share
0,CHEVROLET,VOLT,2017,519,1968.1,0.0231
1,FORD,MUSTANG MACH E,2021,263,1470.4,0.0380
2,TESLA,MODEL S,2016,381,1319.2,0.0787
3,TESLA,MODEL S,2017,306,990.2,0.0523
4,HONDA,HR-V,2023,632,960.1,0.0047
5,ACURA,INTEGRA,2023,196,944.7,0.0153
6,RAM,2500,2018,1861,842.8,0.0118
7,SUBARU,OUTBACK,2020,828,779.5,0.0229
8,CHEVROLET,VOLT,2018,156,768.1,0.0128
9,TESLA,MODEL 3,2018,762,743.4,0.1050


**Interpretation.** This is the integration of the two datasets: complaints from the owner-complaint file divided by CRSS-estimated crash involvement for the same make, model and year. The 2017 Ford Escape is in the 97th percentile, with engine complaints dominating, which is consistent with its publicly reported engine-coolant problems.

## 6.5 Input validation and safeguards

In [7]:
checks = [
    ("narrative shorter than 20 characters", client.post("/api/complaint/analyze", json={"text": "too short"}).status_code),
    ("unknown crash field", client.post("/api/crash/assess", json={"not_a_field": 1}).status_code),
    ("vehicle with no complaints", client.get("/api/vehicles/profile", params={"make": "FORD", "model": "NO SUCH MODEL", "year": 2017}).status_code),
]
display(pd.DataFrame(checks, columns=["request", "HTTP status"]))
empty = client.post("/api/crash/assess", json={}).json()
print("Crash with no inputs ->", empty["warning"])
print("Disclaimer on every response ->", empty["disclaimer"])

,request,HTTP status
0,narrative shorter than 20 characters,422
1,unknown crash field,422
2,vehicle with no complaints,404


Crash with no inputs -> 30 inputs were not provided and were treated as unknown; confidence is reduced.
Disclaimer on every response -> Decision support for human review only. Outputs do not establish causation, liability or the existence of a defect.


**Interpretation.** Invalid requests are rejected with clear errors (422 = invalid input, 404 = not found) instead of producing a misleading prediction. Every response carries the human-review disclaimer, and browser access to the deployed API is restricted to the project's own website.

## 6.6 Automated test suite
The same checks, plus others, run as automated tests before each release.

In [8]:
res = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "--color=no"],
                     cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res.stdout[-600:])

                               [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\starlette\testclient.py:51
  C:\Users\delan\Downloads\NHTSA_Road_Vehicle_Safety_Capstone_Last_Lap\.venv\Lib\site-packages\starlette\testclient.py:51: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
12 passed, 1 warning in 14.30s



## 6.7 Live deployment check
Free hosting puts the API to sleep after 15 minutes without traffic, so the first request can take up to a minute.

In [9]:
import urllib.request
for url in ["https://vehicle-safety-api.onrender.com/health", "https://vehicle-safety-api.onrender.com/api/meta"]:
    try:
        with urllib.request.urlopen(url, timeout=90) as resp:
            print(url, "->", resp.status, resp.read(160).decode())
    except Exception as e:
        print(url, "-> not reachable from this machine right now:", e)

https://vehicle-safety-api.onrender.com/health -> 200 {"status":"ok"}


https://vehicle-safety-api.onrender.com/api/meta -> 200 {"models":[{"task":"S1 Injury crash (CRSS)","model":"Gradient Boosting","headline_metric":"ROC-AUC, 2024 test","headline_value":"0.907","detail":"PR-AUC 0.919 v


## 6.8 Operating, monitoring and retraining
| Concern | Plan |
|---|---|
| **Human oversight** | Outputs are recommendations; every response says they do not establish causation, liability or a defect. |
| **Drift** | Complaint mix changes over time (notebook 02). Track monthly: share of complaints flagged, distribution of recommended components, and agreement with analysts' final routing. Retrain text models yearly, or sooner if agreement falls. |
| **New CRSS years** | Each annual CRSS release: rebuild features with the training-year code lists, re-evaluate on the newest year before replacing models. |
| **Threshold policy** | Thresholds were set on validation data to balance precision and recall; an agency can lower them to catch more cases at the cost of more reviews. |
| **Reproducibility** | Every artifact is rebuilt by the scripts in `src/` (README); raw files are hashed (notebook 02). |
| **Privacy** | The application does not store submitted narratives or crash inputs; they are processed in memory only. The hosting platform keeps standard request logs (time, address, status), not request contents. |